In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv
/kaggle/input/exam-score-prediction-dataset/Exam_Score_Prediction.csv


The first thing I want to do is to discover which features are important 

I will try to use Linear Models 
The models to choose from
Try OLS → Ridge → Lasso → ElasticNet



In [2]:
from sklearn.model_selection import train_test_split

X_full = pd.read_csv("../input/playground-series-s6e1/train.csv")

#X_original_ds = pd.read_csv("../input/exam-score-prediction-dataset/Exam_Score_Prediction.csv")
#X_regen = pd.read_csv("../input/playground-series-s6e1/train.csv")


#print(X_regen.shape)
#print(X_original_ds.shape)

#X_full = pd.concat([X_regen, X_original_ds])


# Dropping id and exam_score. 
# id - useless 
# exam_score - target
y = X_full.exam_score
X_full.drop(["id", "exam_score"], axis=1, inplace=True) 

print(X_full.shape)

### TODO I think we need CV 
X_train, X_valid, y_train, y_valid = train_test_split(X_full, y, test_size = 0.2, random_state=0)

# Checking Cardinality of features
cols = [name for name in X_full.columns if X_full[name].nunique() < 10 and
     X_full[name].dtype == "object"]
print(cols)
print(X_full["course"].nunique())



(630000, 11)
['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']
7


In [3]:
X_full.head(3)

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy
1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate
2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate


In [4]:
#global_mean = X_train["exam_score"].mean()

#X_train.groupby(["age"]).exam_score.agg(["mean", "count"])

- gender - ohe
- course - ohe
- internet_access - ohe
- sleep_quality - ordinal encoding
- study_method - ohe
- facility rating - ordinal encoding
- exam_difficulty - ordinal encoding


X_full.groupby(["internet_access"]).exam_score.mean()

age
17    62.523203
18    62.040926
19    61.683294
20    62.503208
21    62.839442
22    63.159818
23    62.824976
24    62.294657
Name: exam_score, dtype: float64


internet_access
no     62.461274
yes    62.490916
Name: exam_score, dtype: float64


gender
female    62.507099
male      62.176472
other     62.781319
Name: exam_score, dtype: float64

course
b.com      62.476724
b.sc       62.293971
b.tech     62.535864
ba         61.853416
bba        63.219833
bca        62.547552
diploma    62.399600
Name: exam_score, dtype: float64

study_method
coaching         69.310497
group study      60.511131
mixed            65.070324
online videos    59.723746
self-study       57.630206
Name: exam_score, dtype: float64

facility_rating
high      66.684041
low       57.911509
medium    63.039453
Name: exam_score, dtype: float64

exam_difficulty
easy        62.180880
hard        62.718933
moderate    62.577246
Name: exam_score, dtype: float64

sleep_quality
average    62.653817
good       67.858665
poor       56.979379
Name: exam_score, dtype: float64

In [5]:
#result_features = ["study_hours", "class_attendance"]

Testing with all possible features

In [6]:
ohe_columns = ["gender", "course", "study_method", "internet_access"]
ordinal_encoding_columns = ["facility_rating", "exam_difficulty"]
numeric_columns = X_full.drop(ohe_columns + ordinal_encoding_columns + ["sleep_quality"], axis=1, inplace=False).columns.to_list()

In [7]:
def preprocess(df):
    df_temp = df.copy()

    df_temp["feature_formula"] = (
        df_temp['sleep_quality'] / df_temp["sleep_hours"]
    )

    additional_features = ["feature_formula"]
    return df_temp[numeric_columns + additional_features]

In [8]:
def target_encoding_sleep_quality(df):
    mapping = {
        "average": 62.653817,
        "good": 67.858665,
        "poor": 56.979379,
    }

    encoded = df["sleep_quality"].map(mapping)
    encoded = encoded.fillna(56.979379)

    return encoded.to_numpy().reshape(-1, 1)


In [9]:
def target_encoding_internet_access(df):
    mapping = {
        "no": 62.461274,
        "yes": 62.490916,
    }

    encoded = df["internet_access"].map(mapping)
    encoded = encoded.fillna(62.461274)

    return encoded.to_numpy().reshape(-1, 1)


In [10]:
def target_encoding_gender(df):
    mapping = {
        "female": 62.507099,
        "male": 62.176472,
        "other": 62.781319,
    }

    encoded = df["gender"].map(mapping)
    encoded = encoded.fillna(62.507099)

    return encoded.to_numpy().reshape(-1, 1)


In [11]:
def target_encoding_study_method(df):
    mapping = {
        "coaching": 69.310497,
        "group study": 60.511131,
        "mixed": 65.070324,
        "online videos": 59.723746,
        "self-study": 57.630206,
    }

    encoded = df["study_method"].map(mapping)
    encoded = encoded.fillna(60.511131)

    return encoded.to_numpy().reshape(-1, 1)


In [12]:
def target_encoding_facility_rating(df):
    mapping = {
        "high": 66.684041,
        "medium": 63.039453,
        "low": 57.911509,
    }

    encoded = df["facility_rating"].map(mapping)
    encoded = encoded.fillna(63.039453)

    return encoded.to_numpy().reshape(-1, 1)


In [13]:
def target_encoding_exam_difficulty(df):
    mapping = {
        "easy": 62.180880,
        "moderate": 62.577246,
        "hard": 62.718933,
    }

    encoded = df["exam_difficulty"].map(mapping)
    encoded = encoded.fillna(62.577246)

    return encoded.to_numpy().reshape(-1, 1)


In [14]:
def target_encoding_course(df):
    mapping = {
        "b.com": 62.476724,
        "b.sc": 62.293971,
        "b.tech": 62.535864,
        "ba": 61.853416,
        "bba": 63.219833,
        "bca": 62.547552,
        "diploma": 62.399600,
    }

    encoded = df["course"].map(mapping)
    encoded = encoded.fillna(62.399600)

    return encoded.to_numpy().reshape(-1, 1)


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoding_sleep_quality", FunctionTransformer(target_encoding_sleep_quality), ["sleep_quality"]),
        ("target_encoding_age", "passthrough", ["age"]),
        ("target_encoding_internet_access", FunctionTransformer(target_encoding_internet_access), ["internet_access"]),
        ("target_encoding_gender", FunctionTransformer(target_encoding_gender), ["gender"]),
        ("target_encoding_course", OneHotEncoder(), ["course"]), 
        ("target_encoding_study_method", FunctionTransformer(target_encoding_study_method), ["study_method"]), 
        ("target_encoding_facility_rating", FunctionTransformer(target_encoding_facility_rating), ["facility_rating"]), 
        ("target_encoding_exam_difficulty", FunctionTransformer(target_encoding_exam_difficulty), ["exam_difficulty"]),
        ("num", "passthrough", numeric_columns),
    ],
    remainder = "drop"
)

ridge = Ridge()
params = {"alpha": [0.01, 0.1, 1, 3, 10]}
ridge_model = GridSearchCV(ridge, params, cv = 5)

linear_model = LinearRegression()

light_gbm_model = LGBMRegressor()


stacking_reg = StackingRegressor(
    estimators=[
        ("lgb", light_gbm_model),
        ("ridge", ridge_model),
    ],
    final_estimator=RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    passthrough=False,   # set True if you also want raw features in meta-model
)

xgb_model = XGBRegressor(
    random_state = 42,
    n_estimators = 1000, 
    learning_rate = 0.05, 
    n_jobs = 4)


p_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
                   ])

p_xgb.fit(X_train, y_train,
     model__verbose = False)

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('target_encoding_sleep_quality',
                                                  FunctionTransformer(func=<function target_encoding_sleep_quality at 0x7d5af6d49260>),
                                                  ['sleep_quality']),
                                                 ('target_encoding_age',
                                                  'passthrough', ['age']),
                                                 ('target_encoding_internet_access',
                                                  FunctionTransformer(func=<function target_encoding_internet_access at 0x7d...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=1000, n_jobs=4,
                              num_parallel_tree=None, ...))])

In [16]:
from sklearn.metrics import mean_squared_error

pred = p_xgb.predict(X_valid)
y_min = y_valid.min()
y_max = y_valid.max()
pred = np.clip(pred, y_min, y_max)

rmse_xgb = np.sqrt(mean_squared_error(y_valid, pred))

print(rmse_xgb)

8.757342740513637


In [17]:
all_features = ["gender", "age", "course", "study_hours","class_attendance", "internet_access", "sleep_hours", "sleep_quality", "study_method", "facility_rating"]
good_features = ["age", "course", "study_hours","class_attendance", "internet_access", "sleep_hours", "sleep_quality", "study_method", "facility_rating"]
cat_features = ["age", "gender", "exam_difficulty", "course", "internet_access", "sleep_quality", "study_method", "facility_rating"] # TODO age might be cat, might be not


In [18]:
cat_model = CatBoostRegressor(
    depth=8,
    learning_rate=0.03,
    iterations=5000,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)


p1 = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", cat_model)
                   ])

p1.fit(X_train, y_train,)

pred1 = p1.predict(X_valid)
y_min1 = y_valid.min()
y_max1 = y_valid.max()
pred1 = np.clip(pred1, y_min1, y_max1)

rmse_cat = np.sqrt(mean_squared_error(y_valid, pred1))

print(rmse_cat)

8.744290480898398


8.80923730713156 - iterations = 100
8.808428422574382 - iterations = 100, dropped gender, exam_difficulty

In [19]:
np.array(cat_model.get_feature_importance(prettified=True))

array([['15', 44.94981835191134],
       ['16', 19.404389957071647],
       ['0', 10.476519449904046],
       ['11', 9.248270136358368],
       ['12', 7.059641775354125],
       ['17', 5.671294405588906],
       ['1', 0.7873490106038364],
       ['3', 0.48330276620147566],
       ['13', 0.48296775046230617],
       ['2', 0.18122016903710364],
       ['5', 0.18042527762874003],
       ['6', 0.17679768488369627],
       ['7', 0.1671612167021794],
       ['4', 0.14846934097151812],
       ['14', 0.14812804751577224],
       ['10', 0.14809568471633008],
       ['8', 0.14415681731943283],
       ['9', 0.14199215776914145]], dtype=object)

In [20]:
#from catboost import Pool
#pool = Pool(data = X_train, label = y_train, cat_features = cat_features)

#np.array(cat_model.get_feature_importance(
#    pool,
#    "LossFunctionChange",
#    prettified=True
#))

In [21]:
y_pred_xgb = pred
y_pred_linear = pred1

In [22]:
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def hill_climb_weight(y_true, pred1, pred2, init_w=0.5, step=0.05, min_step=1e-3, max_iter=200):
    """
    Simple hill climbing to find best weight w in [0, 1]
    for blend: w*pred1 + (1-w)*pred2
    """
    w = init_w
    best_pred = w * pred1 + (1 - w) * pred2
    best_rmse = rmse(y_true, best_pred)

    for _ in range(max_iter):
        improved = False
        # try moving weight up or down
        for delta in [+step, -step]:
            w_new = w + delta
            # stay in [0,1]
            if w_new < 0 or w_new > 1:
                continue
            blend = w_new * pred1 + (1 - w_new) * pred2
            score = rmse(y_true, blend)

            if score < best_rmse:
                best_rmse = score
                best_pred = blend
                w = w_new
                improved = True

        if not improved:
            # no direction improved: reduce step size
            step /= 2.0
            if step < min_step:
                break

    return w, best_rmse, best_pred


In [23]:
best_w, best_rmse, best_blend_valid = hill_climb_weight(
    y_true=y_valid,
    pred1=y_pred_xgb,
    pred2=y_pred_linear,
    init_w=0.5
)

print("Best blend weight for model 1:", best_w)
print("Best validation RMSE:", best_rmse)

Best blend weight for model 1: 0.3796875
Best validation RMSE: 8.736504043228416


## Linear Regression -- All features 

9.777331710891323 -- All Features After Scaler

9.777331710891323 -- All Features (result_num) No Scaler

10.044434596949047 -- All Features (result_num) Fixed test_train After Scaler

9.76688950837759 -- All Features Fixed test_train After Scaler

9.766889508377577 -- All Features Fixed test_train No Scaler


## Linear Regression -- Chosen features
10.34

10.348836793063448 -- After Scaler

## Ridge (L2) -- Chosen features

10.34883695113191

10.348835824806809 - After Scaler

9.777329850413274 -- All Features No Scaler

9.777329107006 -- All Features Standart Scaler

## LightGBM Regressor

9.1228255959198 -- All Features 

9.103907586662444 -- All Features (result_num) fixed test_train After Scaler

8.81656151530383 -- All Features Fixed test_train No Scaler

8.816472164190326 -- All Features Fixed test_train No Scaler np.clip (LB 8.79173)

## Cat Model

8.759826954739868 

**8.751309236803097**

8.752675158046937 -- All Features Fixed test_train No Scaler np.clip with feature_formula

## XGBoost

8.77656865044218 --  All Features Fixed test_train No Scaler np.clip n_estimator =
500 UNDERFITTING!

8.760711354348857 --  All Features Fixed test_train No Scaler np.clip n_estimator = 1000 (LB 8.72425)

8.760805367687476 -- All Features Fixed test_train No Scaler np.clip n_estimator =
1250 OVERFITTING!

8.761137419139402 --  All Features Fixed test_train No Scaler np.clip n_estimator =
1500 OVERFITTING!

8.76910562992696 --  All Features Fixed test_train No Scaler np.clip n_estimator =
2500 OVERFITTING!

8.809378281772426 --  All Features Fixed test_train No Scaler np.clip n_estimator =
5000 OVERFITTING!

--
8.760218162859363 -- All Features Fixed test_train No Scaler np.clip n_estimator = 1000 with feature_formula

8.76854182585587 -- ChatGPT Version

## HC 

8.744730862311087 -- XGBoost best + Cat Model 8.752675158046937

8.742403576423474 -- XGBoost 8.760218162859363 + Cat Model 8.751309236803097 feature_formula

8.757246091509383 -- Additional DataSet exam-score-prediction-database XGBoost + Cat Model 

8.742494153132299 -- XGBoost 
8.760711354348857 + Cat Model 8.751309236803097 no preprocess


8.742403576423474 -- XGBoost 
8.760218162859363 + Cat Model 8.751309236803097 preprocess, but only feature_formula

8.742494153132299 -- XGBoost 
8.760711354348857 + Cat Model 8.751309236803097 preprocess, all except feature_formula

Remove all except feature_formula

8.743375136103339 -- XGBoost 
8.763351934978504 + Cat Model 8.751309236803097 preprocess, no feature_formuls. df_temp["interaction_1"] = df["study_hours"] * df["class_attendance"]

8.742716605472319 -- XGBoost 
8.761311976654673 + Cat Model 8.751309236803097 preprocess, feature_formuls and interaction_1


Preprocessing only for XGBoost

------
Equall preprocessing for both models

8.754807444306918 -- XGBoost 8.772025473804074 + Cat Model 8.764755099398242 sleep_hours binning

-------
sleep_hours deprecated

8.738660950949726 -- **XGBoost 8.75298259601** + Cat Model 8.747316959756919 Target encoding all categories + feature_formula (LB 8.70545)

8.738387622113555 -- XGBoost 8.7582197691973 + Cat Model 8.74654206266318 Target encoding all categories no feature_formula (LB 8.70111)

8.736715977799092 -- XGBoost 8.757577202668708 + Cat Model 8.744038901747546 Target encoding all categories except age

8.736504043228416 -- XGBoost 8.757342740513637 + Cat Model 8.744290480898398 TE all cats except course (ohe) except age

8.737897282377782 -- XGBoost 8.757342740513637 + Cat Model 8.746465060875373 TE all cats except course, internet_access (ohe) except age

8.736504043228416 -- XGBoost 8.757342740513637 + Cat Model 8.744290480898398 TE all cats except course (ohe), except sleep_quality (ordinal) except age

8.739200581568387 -- XGBoost 8.756547702378779 + Cat Model 8.749017259329795 TE all cats except course (ohe), except sleep_quality (ordinal wrong order) except age

8.73742097353363 -- XGBoost 8.757189788078911 + Cat Model 8.746001493609732 TE all cats except course, gender (ohe), except sleep_quality (ordinal) except age

8.73742097353363 -- XGBoost 8.757189788078911 + Cat Model 8.746001493609732 TE all cats except course, gender (ohe), except sleep_quality, study_method (ordinal) OrdinalEncoder(categories=[["self-study", "online videos", "group study", "mixed","coaching"]]) except age

8.73926008555074 -- XGBoost 8.756161839129952 + Cat Model 8.749101458578192 TE all cats except course, gender, study_method (ohe), except sleep_quality (ordinal) except age (passthrough)

8.73742097353363 -- XGBoost 8.757189788078911 + Cat Model 8.746001493609732 TE all cats except course, gender, study_method (ohe), except sleep_quality, facility_rating (ordinal) except age (passthrough)

8.736827133649967 -- XGBoost 8.756105308738128 + Cat Model 8.745802550061924 TE all cats except course, gender, study_method, facility_rating (ohe), except sleep_quality (ordinal) except age (passthrough)

8.736715977799092 -- XGBoost 8.757577202668708 + Cat Model 8.744038901747546 TE all cats except course, gender, study_method, facility_rating (ohe), except sleep_quality (ordinal), except age (passthrough)

**8.736504043228416 -- XGBoost 8.757342740513637 + Cat Model 8.744290480898398 TE all cats except course(ohe), except age (passthrough)**

-------------------

CatBoost Tuning

Preprocesson - None, random_seed = 42, age is cat, loss = "RMSE"

8.751159302064563 -- XGBoost 8.757342740513637 + Cat Model 8.80923730713156 iterations=100

In [24]:
print(str(best_rmse) + " -- XGBoost " + str(rmse_xgb) + " + Cat Model " + str(rmse_cat))

8.736504043228416 -- XGBoost 8.757342740513637 + Cat Model 8.744290480898398


In [25]:
p_xgb.fit(X_full, y,
     model__verbose = False)
p1.fit(X_full, y,) 


X_test = pd.read_csv("../input/playground-series-s6e1/test.csv")
ids = X_test.id
X_test.drop(["id"], axis=1, inplace=True)


test_pred_xgb = p_xgb.predict(X_test)
test_pred_cat = p1.predict(X_test)

test_pred_blend = best_w * test_pred_xgb + (1 - best_w) * test_pred_cat

y_min = y_valid.min()
y_max = y_valid.max()
pred = np.clip(test_pred_blend, y_min, y_max)

submission_df = pd.DataFrame({"id" : ids, "exam_score": pred})
submission_df.to_csv("sample_submission.csv", index=False)